<a href="https://colab.research.google.com/github/nishthadighe-bit/Data--Engineering-Practicals/blob/main/Practical_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# =====================================================================
# Step 1: Install Apache Airflow & Dependencies
# =====================================================================
print("Installing Apache Airflow...")
!pip install apache-airflow --quiet
print("Apache Airflow installed successfully!")

import os

# Set environment variable to bypass loading heavy example DAGs
os.environ['AIRFLOW__CORE__LOAD_EXAMPLES'] = 'False'

# =====================================================================
# Step 2: Initialize Airflow Database & Create DAG Directory
# =====================================================================
print("\nMigrating and initializing the Airflow Database...")
!airflow db migrate

# Create the DAGs directory inside the Airflow root folder
dag_folder = os.path.expanduser('~/airflow/dags')
os.makedirs(dag_folder, exist_ok=True)
print(f"Airflow DAG directory ready at: {dag_folder}")

# =====================================================================
# Step 3: Write the Updated DAG Script (Airflow 3.x Compatible)
# =====================================================================
dag_code = '''
from datetime import datetime, timedelta
from airflow import DAG
from airflow.providers.standard.operators.bash import BashOperator
from airflow.providers.standard.operators.empty import EmptyOperator

# Default execution arguments
default_args = {
    "owner": "data_engineering_lab",
    "depends_on_past": False,
    "start_date": datetime(2026, 1, 1),
    "email_on_failure": False,
    "email_on_retry": False,
    "retries": 1,
    "retry_delay": timedelta(minutes=5),
}

# Define DAG Context (Using 'schedule' instead of 'schedule_interval')
with DAG(
    "university_etl_orchestration",
    default_args=default_args,
    description="Lab assignment for API and Flat File ETL orchestration",
    schedule="@daily",
    catchup=False,
) as dag:

    # Task 1: Pipeline Init Anchor
    start_pipeline = EmptyOperator(task_id="start_pipeline")

    # Task 2: Simulate Python Extraction Script Execution
    execute_extraction = BashOperator(
        task_id="run_extraction_script",
        bash_command='echo "Executing ETL Data Extraction script..."',
    )

    # Task 3: Finalize and Log Metrics
    pipeline_complete = BashOperator(
        task_id="log_pipeline_success",
        bash_command='echo "ETL Execution completed successfully at $(date)"',
    )

    # Define Task Execution Order
    start_pipeline >> execute_extraction >> pipeline_complete
'''

dag_file_path = os.path.join(dag_folder, 'data_pipeline_dag.py')
with open(dag_file_path, 'w') as f:
    f.write(dag_code.strip())

print(f"DAG script created successfully at '{dag_file_path}'")

# =====================================================================
# Step 4: Test & Verify DAG Tasks via Airflow CLI
# =====================================================================
print("\n=======================================================")
print("RUNNING DAG TASK TESTS VIA AIRFLOW CLI")
print("=======================================================")

print("\n--- Testing Task 1: start_pipeline ---")
!airflow tasks test university_etl_orchestration start_pipeline 2026-01-01

print("\n--- Testing Task 2: run_extraction_script ---")
!airflow tasks test university_etl_orchestration run_extraction_script 2026-01-01

print("\n--- Testing Task 3: log_pipeline_success ---")
!airflow tasks test university_etl_orchestration log_pipeline_success 2026-01-01

Installing Apache Airflow...
Apache Airflow installed successfully!

Migrating and initializing the Airflow Database...
2026-09-15T14:24:45.836149Z [info     ] Performing upgrade to the metadata database [airflow.cli.commands.db_command] loc=db_command.py:134 url=sqlite:////root/airflow/airflow.db
2026-09-15T14:24:45.959869Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-09-15T14:24:45.960153Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-09-15T14:24:45.961882Z [info     ] Migrating the Airflow database [airflow.utils.db] loc=db.py:1189
2026-09-15T14:24:45.968934Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-09-15T14:24:45.969118Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-09-15T14:24:46.002341Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migra